# Initialisation
Code below is for preppring tools for use

---

In [2]:
# import plotting tools
# plotnine is the python clone of ggplot2 from R
# this should be used over plt where possible

import pandas as pd
import matplotlib as plt
from plotnine import *
import numpy as np

In [3]:
from collections import *
from skimage import *
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
from transformers import AutoTokenizer, pipeline, AutoModel
import torch
import re
from thefuzz import process

# Testing
The code below is for the testing of basic premise and ideas, such as probing raw statistics and the like. This may also be used for getting familuar with certain tools and datasets.

In [5]:
df = pd.read_csv('att_sch_stats/att_sch.csv')

attainment_2025 = df[df['DateCode'].astype(str).str.contains("2024-25", na=False)]
above_5_2025 = attainment_2025[
    (
        attainment_2025['Number Of Awards'].astype(str).str.contains("5", na=False)
    ) & (
        attainment_2025['SCQF Level'].astype(str).str.contains("6", na=False)
    )
]
att_list = above_5_2025[["GeographyName", "Value"]]
att_list = att_list.sort_values(by="GeographyName")
att_list = att_list[att_list.GeographyName != "National"]

supressors = ["*", "#"]

supressed_schools = att_list[
    att_list["Value"].astype(str).isin(supressors)
].sort_values(by="GeographyName")

att_list = att_list[
    ~att_list["Value"].astype(str).isin(supressors)
].copy()

att_list

,GeographyName,Value
155091,Aberdeen Grammar School,72
155511,Aboyne Academy,46
171891,Airdrie Academy,35
154111,Alford Academy,41
169511,All Saints Secondary School,38
...,...,...
171261,Whitehill Secondary School,22
151241,Wick High School,18
174131,Williamwood High School,82
174061,Woodfarm High School,74


In [6]:
primary_ys = ["P1", "P2", "P3", "P4", "P5", "P6", "P7"]

sch_sum_2025 = pd.read_excel('sch_sum_stats/sch_sum_2025.xlsx', sheet_name=2, header=1)
hs_sum_2025 = sch_sum_2025[sch_sum_2025["School Type"].astype(str).str.contains("Secondary")]
hs_sum_2025 = hs_sum_2025.drop(columns=primary_ys).drop(columns="Special school pupils")
hs_sum_2025 = hs_sum_2025.sort_values(by="School Name")
hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).isin(["Braeview Academy", "Craigie High School", "Glasgow Gaelic School", "St Joseph's Academy", "The Community School of Auchterarder"])]

supressed_names = supressed_schools["GeographyName"].to_list()
pattern = '|'.join(re.escape(name) for name in supressed_names)
hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).str.contains(pattern, case=False)].copy()

# hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).isin(supressed_schools["GeographyName"])].copy()
# hs_sum_2025 = hs_sum_2025[hs_sum_2025["School Name"].astype(str).isin(att_list["GeographyName"])]
att_list = att_list[att_list["GeographyName"].astype(str).isin(hs_sum_2025["School Name"].astype(str))].copy()
hs_sum_2025 = hs_sum_2025[hs_sum_2025["School Name"].astype(str).isin(att_list["GeographyName"].astype(str))].copy()

In [7]:
target = "Value"

simd_columns = [
    "SIMD Quintile 1 [Note 6]",
    "SIMD Quintile 2 [Note 6]",
    "SIMD Quintile 3 [Note 6]",
    "SIMD Quintile 4 [Note 6]",
    "SIMD Quintile 5 [Note 6]"
]

numeric_columns = simd_columns + ["Pupil roll"]

for column in numeric_columns:
    hs_sum_2025[column] = pd.to_numeric(
        hs_sum_2025[column]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

for column in simd_columns:
    hs_sum_2025[f"{column} proportion"] = (
        hs_sum_2025[column] / hs_sum_2025["Pupil roll"]
    )

simd_features = [
    f"{column} proportion"
    for column in simd_columns
]


In [8]:
URC = ["Urban [Note 7]", "Small town [Note 7]", "Rural [Note 7]"]

def find_midpoint(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip().replace("%","")
    lower, upper = val.split("-")
    return (float(lower) + float(upper)) / 200

for column in URC:
    hs_sum_2025[f"{column} proportion"] = (
        hs_sum_2025[column].apply(find_midpoint)
    )

URF = [
    f"{column} proportion"
    for column in URC
]

In [9]:
denomination = "Denomination of school"

hs_sum_2025["Denomination Encoded"] = (
    hs_sum_2025[denomination]
    .map({
        "Non-denominational school" : 0,
        "Roman Catholic School" : 1
    })
)

In [10]:
features = [
    "Average class size",
    "Pupil roll",
    "Pupil Teacher Ratio",
    "Pupils with an Additional Support Need recorded",
    "Denomination Encoded",
    *simd_features,
    *URF,
]

In [11]:
data = hs_sum_2025.merge(
    att_list[["GeographyName", "Value"]],
    left_on="School Name",
    right_on="GeographyName",
    how="inner",
    validate="one_to_one"
)

x = data[features]
y = pd.to_numeric(data["Value"], errors="coerce")

valid = y.notna()
x = x.loc[valid].copy()
y = y.loc[valid].copy()
numeric_features = features

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features)
])

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

In [13]:
model = Pipeline([
    ("Preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=500,
        random_state=42,
    ))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 8.285910447761195
RMSE: 10.385160118484798
R²: 0.5064382172772146


In [14]:
# import matplotlib.pyplot as plt

# # 1. Calculate average Pupil Teacher Ratio per council
# avg_ptr = db.groupby("Local Authority")["Pupil Teacher Ratio"].mean().reset_index()

# # 2. Create figure and axes object properly
# fig, ax = plt.subplots(figsize=(10, 6))

# # 3. Plot a horizontal bar chart
# ax.barh(avg_ptr["Local Authority"], avg_ptr["Pupil Teacher Ratio"], color="skyblue")

# ax.set_xlabel("Average Pupil Teacher Ratio")
# ax.set_ylabel("Local Authority")
# ax.set_title("Average Pupil Teacher Ratio by Council")

# plt.tight_layout()
# plt.show()
print(hs_sum_2025[numeric_columns].isna().sum())

SIMD Quintile 1 [Note 6]    32
SIMD Quintile 2 [Note 6]    23
SIMD Quintile 3 [Note 6]     7
SIMD Quintile 4 [Note 6]    19
SIMD Quintile 5 [Note 6]    34
Pupil roll                   0
dtype: int64


---

# Testing previous years
starting 2024
Code may be duplicated before possible functions are identified

In [15]:
df = pd.read_csv('att_sch_stats/att_sch.csv')

attainment_2024 = df[df['DateCode'].astype(str).str.contains("2023-24", na=False)]
above_5_2024 = attainment_2024[
    (
        attainment_2024['Number Of Awards'].astype(str).str.contains("5", na=False)
    ) & (
        attainment_2024['SCQF Level'].astype(str).str.contains("6", na=False)
    )
]
att_list_24 = above_5_2024[["GeographyName", "Value"]]
att_list_24 = att_list_24.sort_values(by="GeographyName")
att_list_24 = att_list_24[att_list_24.GeographyName != "National"]

supressors = ["*", "#"]

supressed_schools = att_list_24[
    att_list_24["Value"].astype(str).isin(supressors)
].sort_values(by="GeographyName")

att_list_24 = att_list_24[
    ~att_list_24["Value"].astype(str).isin(supressors)
].copy()

att_list_24= att_list_24.dropna()

In [16]:
primary_ys = ["P1", "P2", "P3", "P4", "P5", "P6", "P7"]

sch_sum_2024 = pd.read_excel('sch_sum_stats/sch_sum_2024.xlsx', sheet_name=2, header=1)
hs_sum_2024 = sch_sum_2024[sch_sum_2024["School Type"].astype(str).str.contains("Secondary")]
hs_sum_2024 = hs_sum_2024.drop(columns=primary_ys).drop(columns="Special school pupils")
hs_sum_2024 = hs_sum_2024.sort_values(by="School Name")
# hs_sum_2024 = hs_sum_2024[~hs_sum_2024["School Name"].astype(str).isin(["Braeview Academy", "Craigie High School", "Glasgow Gaelic School", "St Joseph's Academy", "The Community School of Auchterarder"])]

supressed_names = supressed_schools["GeographyName"].to_list()
pattern = '|'.join(re.escape(name) for name in supressed_names)
hs_sum_2024 = hs_sum_2024[~hs_sum_2024["School Name"].astype(str).str.contains(pattern, case=False)].copy()

# hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).isin(supressed_schools["GeographyName"])].copy()
# hs_sum_2025 = hs_sum_2025[hs_sum_2025["School Name"].astype(str).isin(att_list["GeographyName"])]
att_list_24 = att_list_24[att_list_24["GeographyName"].astype(str).isin(hs_sum_2024["School Name"].astype(str))].copy()
hs_sum_2024 = hs_sum_2024[hs_sum_2024["School Name"].astype(str).isin(att_list_24["GeographyName"].astype(str))].copy()

hs_sum_2024

,Local Authority,SeedCode,School Name,School Type,Teachers (Full Time Equivalent) [Note 1],Pupil roll,Pupil Teacher Ratio,Average class size,Female,Male,...,Small town [Note 7],Rural [Note 7],Urban/rural classification not available [Note 7],Attendance rate (%)[Note 8] [Note 9],Authorised absence rate (%) [Note 8] [Note 9],Unauthorised absence rate (%)[Note 8] [Note 9],Absence due to temporary exclusion rate (%) [Note 8] [Note 9],School's 6-fold urban/rural classification,School's 8-fold urban/rural classification,Denomination of school
6,Aberdeen City,5244439,Aberdeen Grammar School,Secondary,101.2,1310,12.9,0,628,682,...,0-10%,0-10%,0-10%,92.66,5.81,1.5,0.03,Large urban areas,Large urban areas,Non-denominational
19,Aberdeenshire,5257530,Aboyne Academy,Secondary,59.6,659,11.1,0,315,344,...,0-10%,90-100%,0-10%,90.94,6.64,2.41,C,Remote rural areas,Remote rural areas,Non-denominational
27,North Lanarkshire,8505438,Airdrie Academy,Secondary,74.7,1044,14.0,0,533,511,...,0-10%,20-30%,0-10%,83.47,6.18,10.32,0.03,Other urban areas,Other urban areas,Non-denominational
39,Aberdeenshire,5230934,Alford Academy,Secondary,60.6,687,11.3,0,316,371,...,0-10%,90-100%,0-10%,89.83,6.64,3.46,0.06,Remote rural areas,Remote rural areas,Non-denominational
43,Glasgow City,8431930,All Saints Secondary School,Secondary,67.2,911,13.6,0,474,437,...,0-10%,0-10%,0-10%,86.93,7.06,5.99,0.02,Large urban areas,Large urban areas,Roman Catholic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2402,Glasgow City,8439532,Whitehill Secondary School,Secondary,61.4,618,10.1,0,287,331,...,0-10%,0-10%,0-10%,83.37,4.49,12.13,C,Large urban areas,Large urban areas,Non-denominational
2411,Highland,5102138,Wick High School,Secondary,56.5,736,13.0,0,370,366,...,50-60%,40-50%,0-10%,85.7,8.67,5.62,C,Remote small towns,Very Remote Small Towns,Non-denominational
2415,East Renfrewshire,8602034,Williamwood High School,Secondary,140.8,1678,11.9,0,843,835,...,0-10%,0-10%,0-10%,91.95,5.39,2.65,0.01,Large urban areas,Large urban areas,Non-denominational
2428,East Renfrewshire,8601836,Woodfarm High School,Secondary,74.8,701,9.4,0,333,368,...,0-10%,0-10%,0-10%,91.68,6.19,2.11,0.01,Large urban areas,Large urban areas,Non-denominational


In [17]:
# School name columns
att_schools = att_list_24["GeographyName"]
hs_schools = hs_sum_2024["School Name"]

# Schools in att_list but not hs_sum_2024
missing_from_hs = sorted(set(att_schools) - set(hs_schools))

# Schools in hs_sum_2024 but not att_list
missing_from_att = sorted(set(hs_schools) - set(att_schools))

differences = pd.DataFrame({
    "School Name": (
        [*missing_from_hs, *missing_from_att]
    ),
    "Missing From": (
        ["hs_sum_2024"] * len(missing_from_hs)
        + ["att_list"] * len(missing_from_att)
    )
})

display(differences)

,School Name,Missing From


In [18]:
target = "Value"

simd_columns = [
    "SIMD Quintile 1 [Note 6]",
    "SIMD Quintile 2 [Note 6]",
    "SIMD Quintile 3 [Note 6]",
    "SIMD Quintile 4 [Note 6]",
    "SIMD Quintile 5 [Note 6]"
]

numeric_columns = simd_columns + ["Pupil roll"]

for column in numeric_columns:
    hs_sum_2024[column] = pd.to_numeric(
        hs_sum_2024[column]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

for column in simd_columns:
    hs_sum_2024[f"{column} proportion"] = (
        hs_sum_2024[column] / hs_sum_2024["Pupil roll"]
    )

simd_features = [
    f"{column} proportion"
    for column in simd_columns
]


In [19]:
URC = ["Urban [Note 7]", "Small town [Note 7]", "Rural [Note 7]"]

for column in URC:
    hs_sum_2024[f"{column} proportion"] = (
        hs_sum_2024[column].apply(find_midpoint)
    )

URF = [
    f"{column} proportion"
    for column in URC
]

In [20]:
denomination = "Denomination of school"

hs_sum_2024["Denomination Encoded"] = (
    hs_sum_2024[denomination]
    .map({
        "Non-denominational" : 0,
        "Roman Catholic" : 1
    })
)

In [21]:
features = [
    "Average class size",
    "Pupil roll",
    "Pupil Teacher Ratio",
    "Pupils with an Additional Support Need recorded",
    "Denomination Encoded",
    *simd_features,
    *URF,
]

In [22]:
data = hs_sum_2024.merge(
    att_list_24[["GeographyName", "Value"]],
    left_on="School Name",
    right_on="GeographyName",
    how="inner",
    validate="one_to_one"
)

x = data[features]
y = pd.to_numeric(data["Value"], errors="coerce")

valid = y.notna()
x = x.loc[valid].copy()
y = y.loc[valid].copy()
numeric_features = features

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features)
])

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

In [24]:
model = Pipeline([
    ("Preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=500,
        random_state=42,
    ))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 7.946735294117648
RMSE: 9.932631944904692
R²: 0.4814145685661767


---
# Previous year testing 2023
as mentioned above, code repition will be present in the mean time until clean up and organisation is more suitable

In [25]:
df = pd.read_csv('att_sch_stats/att_sch.csv')

attainment_2023 = df[df['DateCode'].astype(str).str.contains("2022-23", na=False)]
above_5_2023 = attainment_2023[
    (
        attainment_2023['Number Of Awards'].astype(str).str.contains("5", na=False)
    ) & (
        attainment_2023['SCQF Level'].astype(str).str.contains("6", na=False)
    )
]
att_list_23 = above_5_2023[["GeographyName", "Value"]]
att_list_23 = att_list_23.sort_values(by="GeographyName")
att_list_23 = att_list_23[att_list_23.GeographyName != "National"]

supressors = ["*", "#"]

supressed_schools = att_list_23[
    att_list_23["Value"].astype(str).isin(supressors)
].sort_values(by="GeographyName")

att_list_23 = att_list_23[
    ~att_list_23["Value"].astype(str).isin(supressors)
].copy()

att_list_23 = att_list_23.dropna()

In [26]:
primary_ys = ["P1", "P2", "P3", "P4", "P5", "P6", "P7"]

sch_sum_2023 = pd.read_excel('sch_sum_stats/sch_sum_2023.xlsx', sheet_name=2, header=1)
hs_sum_2023 = sch_sum_2023[sch_sum_2023["School Type"].astype(str).str.contains("Secondary")]
hs_sum_2023 = hs_sum_2023.drop(columns=primary_ys).drop(columns="Special school pupils")
hs_sum_2023 = hs_sum_2023.sort_values(by="School Name")
# hs_sum_2024 = hs_sum_2024[~hs_sum_2024["School Name"].astype(str).isin(["Braeview Academy", "Craigie High School", "Glasgow Gaelic School", "St Joseph's Academy", "The Community School of Auchterarder"])]

supressed_names = supressed_schools["GeographyName"].to_list()
pattern = '|'.join(re.escape(name) for name in supressed_names)
hs_sum_2023 = hs_sum_2023[~hs_sum_2023["School Name"].astype(str).str.contains(pattern, case=False)].copy()

# hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).isin(supressed_schools["GeographyName"])].copy()
# hs_sum_2025 = hs_sum_2025[hs_sum_2025["School Name"].astype(str).isin(att_list["GeographyName"])]
att_list_23 = att_list_23[att_list_23["GeographyName"].astype(str).isin(hs_sum_2023["School Name"].astype(str))].copy()
hs_sum_2023 = hs_sum_2023[hs_sum_2023["School Name"].astype(str).isin(att_list_23["GeographyName"].astype(str))].copy()

hs_sum_2023

,Local Authority,SeedCode,School Name,School Type,Teachers (Full Time Equivalent) [Note 1],Pupil roll,Pupil Teacher Ratio,Average class size,Female,Male,...,Small town [Note 7],Rural [Note 7],Urban/rural classification not available [Note 7],Attendance rate (%)[Note 8] [Note 9],Authorised absence rate (%) [Note 8] [Note 9],Unauthorised absence rate (%)[Note 8] [Note 9],Absence due to temporary exclusion rate (%) [Note 8] [Note 9],School's 6-fold urban/rural classification,School's 8-fold urban/rural classification,Denomination of school
6,Aberdeen City,5244439,Aberdeen Grammar School,Secondary,107.4,1205,11.2,0,567,638,...,0-10%,0-10%,0-10%,92.04,5.88,2.05,0.03,Large urban areas,Large urban areas,Non-denominational
18,Aberdeenshire,5257530,Aboyne Academy,Secondary,60.6,669,11.0,0,320,349,...,0-10%,90-100%,0-10%,92.26,5.67,c,c,Remote rural areas,Remote rural areas,Non-denominational
26,North Lanarkshire,8505438,Airdrie Academy,Secondary,73.9,1032,14.0,0,533,499,...,0-10%,20-30%,0-10%,82.98,5.84,11.14,0.04,Other urban areas,Other urban areas,Non-denominational
38,Aberdeenshire,5230934,Alford Academy,Secondary,59.0,733,12.4,0,344,389,...,0-10%,90-100%,0-10%,89.59,6.42,3.92,0.07,Remote rural areas,Remote rural areas,Non-denominational
42,Glasgow City,8431930,All Saints Secondary School,Secondary,67.6,905,13.4,0,461,444,...,0-10%,0-10%,0-10%,89.22,5.82,4.94,0.02,Large urban areas,Large urban areas,Roman Catholic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2408,West Lothian,5510430,Whitburn Academy,Secondary,64.1,851,13.3,0,419,432,...,30-40%,0-10%,0-10%,82.32,13.33,4.34,0.02,Other urban areas,Other urban areas,Non-denominational
2413,Glasgow City,8439532,Whitehill Secondary School,Secondary,56.4,604,10.7,0,264,340,...,0-10%,0-10%,0-10%,87.07,4.82,c,c,Large urban areas,Large urban areas,Non-denominational
2426,East Renfrewshire,8602034,Williamwood High School,Secondary,143.8,1668,11.6,0,849,819,...,0-10%,0-10%,0-10%,92.34,5.01,2.64,0.01,Large urban areas,Large urban areas,Non-denominational
2439,East Renfrewshire,8601836,Woodfarm High School,Secondary,70.8,693,9.8,0,339,354,...,0-10%,0-10%,0-10%,91.79,6.46,c,c,Large urban areas,Large urban areas,Non-denominational


In [27]:
# School name columns
att_schools = att_list_23["GeographyName"]
hs_schools = hs_sum_2023["School Name"]

# Schools in att_list but not hs_sum_2023
missing_from_hs = sorted(set(att_schools) - set(hs_schools))

# Schools in hs_sum_2023 but not att_list
missing_from_att = sorted(set(hs_schools) - set(att_schools))

differences = pd.DataFrame({
    "School Name": (
        [*missing_from_hs, *missing_from_att]
    ),
    "Missing From": (
        ["hs_sum_2023"] * len(missing_from_hs)
        + ["att_list"] * len(missing_from_att)
    )
})

display(differences)

,School Name,Missing From


In [28]:
target = "Value"

simd_columns = [
    "SIMD Quintile 1 [Note 6]",
    "SIMD Quintile 2 [Note 6]",
    "SIMD Quintile 3 [Note 6]",
    "SIMD Quintile 4 [Note 6]",
    "SIMD Quintile 5 [Note 6]"
]

numeric_columns = simd_columns + ["Pupil roll"]

for column in numeric_columns:
    hs_sum_2023[column] = pd.to_numeric(
        hs_sum_2023[column]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

for column in simd_columns:
    hs_sum_2023[f"{column} proportion"] = (
        hs_sum_2023[column] / hs_sum_2023["Pupil roll"]
    )

simd_features = [
    f"{column} proportion"
    for column in simd_columns
]


In [29]:
denomination = "Denomination of school"

hs_sum_2023["Denomination Encoded"] = (
    hs_sum_2023[denomination]
    .map({
        "Non-denominational" : 0,
        "Roman Catholic" : 1
    })
)

In [30]:
URC = ["Urban [Note 7]", "Small town [Note 7]", "Rural [Note 7]"]

for column in URC:
    hs_sum_2023[f"{column} proportion"] = (
        hs_sum_2023[column].apply(find_midpoint)
    )

URF = [
    f"{column} proportion"
    for column in URC
]

In [31]:
features = [
    "Average class size",
    "Pupil roll",
    "Pupil Teacher Ratio",
    "Pupils with an Additional Support Need recorded",
    "Denomination Encoded",
    *simd_features,
    *URF,
]

In [32]:
data = hs_sum_2023.merge(
    att_list_23[["GeographyName", "Value"]],
    left_on="School Name",
    right_on="GeographyName",
    how="inner",
    validate="one_to_one"
)

x = data[features]
y = pd.to_numeric(data["Value"], errors="coerce")

valid = y.notna()
x = x.loc[valid].copy()
y = y.loc[valid].copy()
numeric_features = features

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features)
])

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

In [34]:
model = Pipeline([
    ("Preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=500,
        random_state=42,
    ))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 9.847373134328357
RMSE: 11.647260752845543
R²: 0.42337892083734185


---
# Previous Year Testing 2022
as before expect code repition until clean up

In [35]:
df = pd.read_csv('att_sch_stats/att_sch.csv')

attainment_2022 = df[df['DateCode'].astype(str).str.contains("2021-22", na=False)]
above_5_2022 = attainment_2022[
    (
        attainment_2022['Number Of Awards'].astype(str).str.contains("5", na=False)
    ) & (
        attainment_2022['SCQF Level'].astype(str).str.contains("6", na=False)
    )
]
att_list_22 = above_5_2022[["GeographyName", "Value"]]
att_list_22 = att_list_22.sort_values(by="GeographyName")
att_list_22 = att_list_22[att_list_22.GeographyName != "National"]

supressors = ["*", "#"]

supressed_schools = att_list_22[
    att_list_22["Value"].astype(str).isin(supressors)
].sort_values(by="GeographyName")

att_list_22 = att_list_22[
    ~att_list_22["Value"].astype(str).isin(supressors)
].copy()

att_list_22 = att_list_22.dropna()
att_list_22

,GeographyName,Value
79770,Aberdeen Grammar School,59
80190,Aboyne Academy,46
96570,Airdrie Academy,36
78790,Alford Academy,49
94190,All Saints Secondary School,37
...,...,...
95940,Whitehill Secondary School,21
75920,Wick High School,20
98810,Williamwood High School,73
98740,Woodfarm High School,78


In [36]:
primary_ys = ["P1", "P2", "P3", "P4", "P5", "P6", "P7"]

sch_sum_2022 = pd.read_excel('sch_sum_stats/sch_sum_2022.xlsx', sheet_name=2, header=1)
hs_sum_2022 = sch_sum_2022[sch_sum_2022["School Type"].astype(str).str.contains("Secondary")]
hs_sum_2022 = hs_sum_2022.drop(columns=primary_ys).drop(columns="Special school pupils")
hs_sum_2022 = hs_sum_2022.sort_values(by="School Name")
# hs_sum_2024 = hs_sum_2024[~hs_sum_2024["School Name"].astype(str).isin(["Braeview Academy", "Craigie High School", "Glasgow Gaelic School", "St Joseph's Academy", "The Community School of Auchterarder"])]

supressed_names = supressed_schools["GeographyName"].to_list()
pattern = '|'.join(re.escape(name) for name in supressed_names)
hs_sum_2022 = hs_sum_2022[~hs_sum_2022["School Name"].astype(str).str.contains(pattern, case=False)].copy()

# hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).isin(supressed_schools["GeographyName"])].copy()
# hs_sum_2025 = hs_sum_2025[hs_sum_2025["School Name"].astype(str).isin(att_list["GeographyName"])]
att_list_22 = att_list_22[att_list_22["GeographyName"].astype(str).isin(hs_sum_2022["School Name"].astype(str))].copy()
hs_sum_2022 = hs_sum_2022[hs_sum_2022["School Name"].astype(str).isin(att_list_22["GeographyName"].astype(str))].copy()

hs_sum_2022

,Local Authority,SeedCode,School Name,School Type,Teachers (Full Time Equivalent) [Note 1],Pupil roll,Pupil Teacher Ratio,Average class size,Female,Male,...,Small town [Note 7],Rural [Note 7],Urban/rural classification not available [Note 7],Attendance rate (%)[Note 8] [Note 9],Authorised absence rate (%) [Note 8] [Note 9],Unauthorised absence rate (%)[Note 8] [Note 9],Absence due to temporary exclusion rate (%) [Note 8] [Note 9],School's 6-fold urban/rural classification,School's 8-fold urban/rural classification,Denomination of school
6,Aberdeen City,5244439,Aberdeen Grammar School,Secondary,90.2,1173,13.0,z,566,607,...,0-10%,0-10%,0-10%,89.78,3.86,6.34,0.02,Large urban areas,Large urban areas,Non-denominational
18,Aberdeenshire,5257530,Aboyne Academy,Secondary,61.1,707,11.6,z,344,363,...,0-10%,90-100%,0-10%,93.22,5.54,1.19,0.05,Remote rural areas,Remote rural areas,Non-denominational
27,North Lanarkshire,8505438,Airdrie Academy,Secondary,79.7,1053,13.2,z,530,523,...,0-10%,10-20%,0-10%,82.52,5.55,11.89,0.04,Other urban areas,Other urban areas,Non-denominational
39,Aberdeenshire,5230934,Alford Academy,Secondary,60.5,704,11.6,z,337,367,...,0-10%,90-100%,0-10%,92.27,5.24,c,c,Remote rural areas,Remote rural areas,Non-denominational
43,Glasgow City,8431930,All Saints Secondary School,Secondary,73.2,874,11.9,z,440,434,...,0-10%,0-10%,0-10%,89.16,3.97,6.87,0.01,Large urban areas,Large urban areas,Roman Catholic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2419,Glasgow City,8439532,Whitehill Secondary School,Secondary,63.8,603,9.5,z,255,348,...,0-10%,0-10%,0-10%,90.62,4.26,5.08,0.05,Large urban areas,Large urban areas,Non-denominational
2428,Highland,5102138,Wick High School,Secondary,57.2,731,12.8,z,364,367,...,50-60%,40-50%,0-10%,86.76,8.14,5.07,0.04,Remote small towns,Very Remote Small Towns,Non-denominational
2432,East Renfrewshire,8602034,Williamwood High School,Secondary,131.4,1648,12.5,z,837,811,...,0-10%,0-10%,0-10%,94.49,3.86,c,c,Large urban areas,Large urban areas,Non-denominational
2445,East Renfrewshire,8601836,Woodfarm High School,Secondary,69.8,694,9.9,z,343,351,...,0-10%,0-10%,0-10%,91.39,5.78,c,c,Large urban areas,Large urban areas,Non-denominational


In [37]:
# School name columns
att_schools = att_list_22["GeographyName"]
hs_schools = hs_sum_2022["School Name"]

# Schools in att_list but not hs_sum_2023
missing_from_hs = sorted(set(att_schools) - set(hs_schools))

# Schools in hs_sum_2023 but not att_list
missing_from_att = sorted(set(hs_schools) - set(att_schools))

differences = pd.DataFrame({
    "School Name": (
        [*missing_from_hs, *missing_from_att]
    ),
    "Missing From": (
        ["hs_sum_2022"] * len(missing_from_hs)
        + ["att_list"] * len(missing_from_att)
    )
})

display(differences)

,School Name,Missing From


In [38]:
target = "Value"

simd_columns = [
    "SIMD Quintile 1 [Note 6]",
    "SIMD Quintile 2 [Note 6]",
    "SIMD Quintile 3 [Note 6]",
    "SIMD Quintile 4 [Note 6]",
    "SIMD Quintile 5 [Note 6]"
]

numeric_columns = simd_columns + ["Pupil roll"]

for column in numeric_columns:
    hs_sum_2022[column] = pd.to_numeric(
        hs_sum_2022[column]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

for column in simd_columns:
    hs_sum_2022[f"{column} proportion"] = (
        hs_sum_2022[column] / hs_sum_2022["Pupil roll"]
    )

simd_features = [
    f"{column} proportion"
    for column in simd_columns
]


In [ ]:
denomination = "Denomination of school"

hs_sum_2022["Denomination Encoded"] = (
    hs_sum_2022[denomination]
    .map({
        "Non-denominational" : 0,
        "Roman Catholic" : 1
    })
)

In [40]:
URC = ["Urban [Note 7]", "Small town [Note 7]", "Rural [Note 7]"]

for column in URC:
    hs_sum_2022[f"{column} proportion"] = (
        hs_sum_2022[column].apply(find_midpoint)
    )

URF = [
    f"{column} proportion"
    for column in URC
]

In [41]:
features = [
    # "Average class size",
    "Pupil roll",
    "Pupil Teacher Ratio",
    "Pupils with an Additional Support Need recorded",
    "Denomination Encoded",
    *simd_features,
    *URF,
]

In [42]:
data = hs_sum_2022.merge(
    att_list_22[["GeographyName", "Value"]],
    left_on="School Name",
    right_on="GeographyName",
    how="inner",
    validate="one_to_one"
)

x = data[features]
y = pd.to_numeric(data["Value"], errors="coerce")

valid = y.notna()
x = x.loc[valid].copy()
y = y.loc[valid].copy()
numeric_features = features

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features)
])

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

In [44]:
model = Pipeline([
    ("Preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=500,
        random_state=42,
    ))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 7.382794117647059
RMSE: 10.325370481887576
R²: 0.5769198274654657


---
# Previous Year Testing 2021
NOTE: 2020 and 2021 were years affected by covid, so there will be a dip in the overall attainment and attendance

In [55]:
df = pd.read_csv('att_sch_stats/att_sch.csv')

attainment_2021 = df[df['DateCode'].astype(str).str.contains("2020-21", na=False)]
above_5_2021 = attainment_2021[
    (
        attainment_2021['Number Of Awards'].astype(str).str.contains("5", na=False)
    ) & (
        attainment_2021['SCQF Level'].astype(str).str.contains("6", na=False)
    )
]
att_list_21 = above_5_2021[["GeographyName", "Value"]]
att_list_21 = att_list_21.sort_values(by="GeographyName")
att_list_21 = att_list_21[att_list_21.GeographyName != "National"]

supressors = ["*", "#"]

supressed_schools = att_list_21[
    att_list_21["Value"].astype(str).isin(supressors)
].sort_values(by="GeographyName")

att_list_21 = att_list_21[
    ~att_list_21["Value"].astype(str).isin(supressors)
].copy()

att_list_21 = att_list_21.dropna()
att_list_21

,GeographyName,Value
54640,Aberdeen Grammar School,61
55060,Aboyne Academy,47
71440,Airdrie Academy,42
53660,Alford Academy,52
69060,All Saints Secondary School,32
...,...,...
70810,Whitehill Secondary School,28
50790,Wick High School,28
73680,Williamwood High School,77
73610,Woodfarm High School,73


In [58]:
primary_ys = ["P1", "P2", "P3", "P4", "P5", "P6", "P7"]

sch_sum_2021 = pd.read_excel('sch_sum_stats/sch_sum_2021.xlsx', sheet_name=2, header=1)
hs_sum_2021 = sch_sum_2021[sch_sum_2021["School Type"].astype(str).str.contains("Secondary")]
hs_sum_2021 = hs_sum_2021.drop(columns=primary_ys).drop(columns="Special school pupils")
hs_sum_2021 = hs_sum_2021.sort_values(by="School Name")
# hs_sum_2024 = hs_sum_2024[~hs_sum_2024["School Name"].astype(str).isin(["Braeview Academy", "Craigie High School", "Glasgow Gaelic School", "St Joseph's Academy", "The Community School of Auchterarder"])]

supressed_names = supressed_schools["GeographyName"].to_list()
pattern = '|'.join(re.escape(name) for name in supressed_names)
hs_sum_2021 = hs_sum_2021[~hs_sum_2021["School Name"].astype(str).str.contains(pattern, case=False)].copy()

# hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).isin(supressed_schools["GeographyName"])].copy()
# hs_sum_2025 = hs_sum_2025[hs_sum_2025["School Name"].astype(str).isin(att_list["GeographyName"])]
att_list_21 = att_list_21[att_list_21["GeographyName"].astype(str).isin(hs_sum_2021["School Name"].astype(str))].copy()
hs_sum_2021 = hs_sum_2021[hs_sum_2021["School Name"].astype(str).isin(att_list_21["GeographyName"].astype(str))].copy()

hs_sum_2021

,Local Authority,SeedCode,School Name,School Type,Teachers (Full Time Equivalent) [Note 1],Pupil roll,Pupil Teacher Ratio,Average class size,Female,Male,...,Small town,Rural,Urban/rural classification not available,Attendance rate,Authorised absence rate,Unauthorised absence rate,Temporary exclusion rate,School's 6-fold urban/rural classification,School's 8-fold urban/rural classification,Denomination of school
48,Aberdeen City,5244439,Aberdeen Grammar School,Secondary,92.7,1163,12.5,0.0,559,604,...,0-10%,0-10%,0-10%,89.78,3.86,6.34,0.02,Large urban areas,Large urban areas,Non-denominational
49,Aberdeenshire,5257530,Aboyne Academy,Secondary,64.5,724,11.2,0.0,349,375,...,0-10%,90-100%,0-10%,93.22,5.54,1.19,0.05,Remote rural areas,Remote rural areas,Non-denominational
50,North Lanarkshire,8505438,Airdrie Academy,Secondary,74.5,1039,13.9,0.0,509,530,...,0-10%,20-30%,0-10%,82.52,5.55,11.89,0.04,Other urban areas,Other urban areas,Non-denominational
52,Aberdeenshire,5230934,Alford Academy,Secondary,62.0,711,11.5,0.0,340,371,...,0-10%,90-100%,0-10%,92.27,5.24,c,c,Remote rural areas,Remote rural areas,Non-denominational
53,Glasgow City,8431930,All Saints Secondary School,Secondary,71.6,851,11.9,0.0,432,419,...,0-10%,0-10%,0-10%,89.16,3.97,6.87,0.01,Large urban areas,Large urban areas,Roman Catholic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2451,Glasgow City,8439532,Whitehill Secondary School,Secondary,61.2,564,9.2,0.0,236,328,...,0-10%,0-10%,0-10%,90.62,4.26,5.08,0.05,Large urban areas,Large urban areas,Non-denominational
2452,Highland,5102138,Wick High School,Secondary,51.7,752,14.5,0.0,376,376,...,50-60%,30-40%,0-10%,86.76,8.14,5.07,0.04,Remote small towns,Very Remote Small Towns,Non-denominational
2453,East Renfrewshire,8602034,Williamwood High School,Secondary,140.6,1654,11.8,0.0,823,831,...,0-10%,0-10%,0-10%,94.49,3.86,c,c,Large urban areas,Large urban areas,Non-denominational
2454,East Renfrewshire,8601836,Woodfarm High School,Secondary,67.0,715,10.7,0.0,356,359,...,0-10%,0-10%,0-10%,91.39,5.78,c,c,Large urban areas,Large urban areas,Non-denominational


In [59]:
# School name columns
att_schools = att_list_21["GeographyName"]
hs_schools = hs_sum_2021["School Name"]

# Schools in att_list but not hs_sum_2023
missing_from_hs = sorted(set(att_schools) - set(hs_schools))

# Schools in hs_sum_2023 but not att_list
missing_from_att = sorted(set(hs_schools) - set(att_schools))

differences = pd.DataFrame({
    "School Name": (
        [*missing_from_hs, *missing_from_att]
    ),
    "Missing From": (
        ["hs_sum_2022"] * len(missing_from_hs)
        + ["att_list"] * len(missing_from_att)
    )
})

display(differences)

,School Name,Missing From


In [60]:
target = "Value"

simd_columns = [
    "SIMD Quintile 1",
    "SIMD Quintile 2",
    "SIMD Quintile 3",
    "SIMD Quintile 4",
    "SIMD Quintile 5"
]

numeric_columns = simd_columns + ["Pupil roll"]

for column in numeric_columns:
    hs_sum_2021[column] = pd.to_numeric(
        hs_sum_2021[column]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

for column in simd_columns:
    hs_sum_2021[f"{column} proportion"] = (
        hs_sum_2021[column] / hs_sum_2021["Pupil roll"]
    )

simd_features = [
    f"{column} proportion"
    for column in simd_columns
]


In [61]:
denomination = "Denomination of school"

hs_sum_2021["Denomination Encoded"] = (
    hs_sum_2021[denomination]
    .map({
        "Non-denominational" : 0,
        "Roman Catholic" : 1
    })
)

In [62]:
URC = ["Urban", "Small town", "Rural"]

for column in URC:
    hs_sum_2021[f"{column} proportion"] = (
        hs_sum_2021[column].apply(find_midpoint)
    )

URF = [
    f"{column} proportion"
    for column in URC
]

In [63]:
features = [
    "Average class size",
    "Pupil roll",
    "Pupil Teacher Ratio",
    "Pupils with an Additional Support Need recorded",
    "Denomination Encoded",
    *simd_features,
    *URF,
]

In [64]:
data = hs_sum_2021.merge(
    att_list_21[["GeographyName", "Value"]],
    left_on="School Name",
    right_on="GeographyName",
    how="inner",
    validate="one_to_one"
)

x = data[features]
y = pd.to_numeric(data["Value"], errors="coerce")

valid = y.notna()
x = x.loc[valid].copy()
y = y.loc[valid].copy()
numeric_features = features

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features)
])

In [65]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

In [66]:
model = Pipeline([
    ("Preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=500,
        random_state=42,
    ))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 8.97029411764706
RMSE: 11.515692398278304
R²: 0.2736345789681295


It should be noted here that while a drop in the correlation given by the r^2 value was expected, a drop of this size was not. Yes the pandemic did have an adverse effect on all pupils, enough so that the similar results which we have seen so far show that the pandemic had a strong influence on not just those least advantaged.
It again should be mentioned that basic experimentation has yet to be carried out and is purely from the base model statistical output. It can be imagined that attendance would be a larger factor to this and is something that can be looked into -> note that as a child experience a lower SIMD score, the more likely they are to have a high absence rate from education having a following knock on effect not just from the pandemic but lack of participation in learning.